**What is AI agents**
--it is a large language model(LLM)
--An ai agents is an LLM that can
  - use tools
  - take multiple steps
  - decide which action to take
  - act on the real world

unlike chatbot that can only answer from memory, agents can search, query databases, run code, and call APIs

**chatbot** -- only have the knowledge and no potential t work on with
**LLM**     -- has both knowledge and potential to with also

**Regular chatbot** : A friend who answers from memory

**Ai agents** : a personal assistant who checks you calendar, searches online, sends an email, and calls a vendor - all from one request

The difference : tools + multi-step action

**Tool calling**
it means giving the AI the ability to run specific python functions. you define the tools. The AI decides when to call them

**get_schema()**     - returns the database structure to the AI so it knows column names and types
**generate_sql(question)** - sends the user question + schema to Groq LLM, returns a SQL query string
**execute_sql(query)**  - runs the AI- generated SQL on SQLite and returns the results as a DataFrame

**Ai agents** : a personal assistant who checks you calendar, searches online, sends an email, and calls a vendor - all from one request

# **AI AGENT**

**Components:**

**LLM** - Brain / Decision Maker (Ex: Groq's LLaMa model)

**Tools** - Functions it can call (Ex: SQL executer, web search)

**Memory** - Conversation History (Ex: Chat History)

**Reasoning** - Multi-step planning (Ex: Chain of thought)

Unlike chatbots that only answer from memory, agents can search, query databases, run code and call APIs

**Text-to-SQL:**

Stage 1: (Schema Injection) Tell the LLm what the database looks like --table name, column names, data types, sample rows.

Stage 2: (SQL Generation) LLM reads schema + user question and wrties a valid SQL query temperature = 0.0 for exact output.

Stage 3: (Execute + Respond) Python runs the SQL on SQLite, returns result. LLM writes a plain-English answer.

**Complete Text-to-SQL Pipeline**

1. Load a csv into SQLite

2. Get Database Schema

3. Generate SQL with Groq

4. Execue SQL on SQLite

5. Natural Language Answer



In [ ]:
!pip install groq -q
print("Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00
Libraries installed successfully


In [ ]:
import sqlite3
import pandas as pd
import os
from groq import Groq

In [ ]:
os.environ["GROQ_API_KEY"] = "gsk_EcoTePgPu7Z6xS814LtuWGdyb3FYkclL5nQAUvBpKbxtCsKaQy"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using model: {MODEL}")


Groq client initialized successfully
Using model: llama-3.1-8b-instant


In [ ]:
import io

csv_data = csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
df = pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows")
df.head()

Dataset loaded: 30 rows, 8 columns

First 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
conn = sqlite3.connect("college.db")

df.to_sql("students", conn, if_exists="replace", index=False)
print("Database created: college.db")
print("Table 'students' created with 30 student records")

test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"\nVerification: {test_df['total_rows'][0]} rows in database")

Database created: college.db
Table 'students' created with 30 student records

Verification: 30 rows in database


In [ ]:
def get_schema(conn, table_name="students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns:")
  for col in columns:
    schema_lines.append(f" - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()
  schema_lines.append("\nSample Rows:")
  for row in sample_rows:
    schema_lines.append(f" - {row}")

  return "\n".join(schema_lines)
schema=get_schema(conn)
print(schema)

Table: students
Columns:
 - student_id (INTEGER)
 - name (TEXT)
 - age (INTEGER)
 - gender (TEXT)
 - subject (TEXT)
 - marks (INTEGER)
 - attendance (INTEGER)
 - grade (TEXT)

Sample Rows:
 - (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 - (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 - (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question, schema_text, client, model):
  system_prompt = f"""You are an expert SQL assistant.
  You are connected to a SQLite database with the following structure:
  {schema_text}

  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or text - only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is: students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses (example: WHERE subjecy = 'Programming')
  7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """

  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature=0.0
  )

  sql_query = response.choices[0].message.content.strip()
  return sql_query
question = "Show me all female students"
print(f"Question: {question}")
print("\nGenerating SQL...")

sql = generate_sql(question, schema, client, MODEL)
print(f"Generated SQL:\n{sql}")


Question: Show me all female students

Generating SQL...
Generated SQL:
SELECT * FROM students WHERE gender = 'Female'


In [ ]:
import re

def execute_sql(sql_query,conn):
  """clean the ai-generated sql and execute it on the sqlite database
  return the results as pandas dataframe.

  parameters:"""

  clean_sql=sql_query.strip()
  clean_sql=re.sub(r'``sql\s*','',clean_sql)
  clean_sql=clean_sql.strip()
  try:
    result_df=pd.read_sql_query(clean_sql,conn)
    return result_df,None
  except Exception as e:
    return None,str(e)
print("Executing sql: ",sql)
result,error=execute_sql(sql,conn)
if error:
    print("Error: ",error)
else:
    print("query returned: ",len(result),"rows")
    print(result)

Executing sql:  SELECT * FROM students WHERE gender = 'Female'
query returned:  15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female

In [ ]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  """
  the main ai agent function,
  takes a user question in plain english and returns database results
  """
  print("="*60)
  print("user question: ",user_question)
  print("="*6)

  #step1:get the database schema
  if verbose:
    print("[step 1]:reading database schema .....")
  schema_text=get_schema(conn)
  if verbose:
    print("schema loaded sucessfully")
  if verbose:
    print("[step 2]:generating sql query.....")

  generated_sql=generate_sql(user_question,schema_text,client,model)
  if verbose:
    print("generated sql: ",generated_sql)
  if verbose:
    print("[step 3]:executing sql on the database...")
  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print("sql execution error: ",error)
    return None,generated_sql
  if verbose:
    print("[step 4] query returned",len(result_df),"rows")
    print("\nresult: ")
    print("-"*80)
    print(result_df.to_string(index=False))
    print("-"*80)
    return result_df,generated_sql
result,sql_used=text_to_sql_agent(
    "show top 5 students in programming",
    conn,client,MODEL
)

user question:  show top 5 students in programming
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT name, marks, attendance, grade FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
[step 3]:executing sql on the database...
[step 4] query returned 5 rows

result: 
--------------------------------------------------------------------------------
        name  marks  attendance grade
Aditya Kumar     97          99    A+
 Rohan Mehta     95          98    A+
Anjali Singh     94          97    A+
 Nandita Rao     92          96    A+
  Arjun Nair     91          94    A+
--------------------------------------------------------------------------------


In [ ]:
result1, _ = text_to_sql_agent(
    "Show mw all students who study Mathematics",
    conn,
    client,
    MODEL
)

user question:  Show mw all students who study Mathematics
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE subject = 'Mathematics'
[step 3]:executing sql on the database...
[step 4] query returned 10 rows

result: 
--------------------------------------------------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
       

In [ ]:
result2, _ = text_to_sql_agent(
    "What is the average marks for each subject?",
    conn,
    client,
    MODEL
)


user question:  What is the average marks for each subject?
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT subject, AVG(marks) FROM students GROUP BY subject
[step 3]:executing sql on the database...
[step 4] query returned 3 rows

result: 
--------------------------------------------------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4
--------------------------------------------------------------------------------


In [ ]:
result3, _ = text_to_sql_agent(
    "Show students who scored more thatn 90 marks?",
    conn,
    client,
    MODEL
)


user question:  Show students who scored more thatn 90 marks?
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE marks > 90
[step 3]:executing sql on the database...
[step 4] query returned 6 rows

result: 
--------------------------------------------------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+
--------------------------------------------------------------------------------


In [ ]:
result4, _ = text_to_sql_agent(
    "How many male and female students are there?",
    conn, client, MODEL
)

user question:  How many male and female students are there?
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students
[step 3]:executing sql on the database...
[step 4] query returned 1 rows

result: 
--------------------------------------------------------------------------------
 male_count  female_count
         15            15
--------------------------------------------------------------------------------


In [ ]:
result5, _ = text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming, ordered by marks",
    conn, client, MODEL
)

user question:  Show female students who scored above 85 in Science or Programming, ordered by marks
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE gender = 'Female' AND subject IN ('Science', 'Programming') AND marks > 85 ORDER BY marks DESC
[step 3]:executing sql on the database...
[step 4] query returned 5 rows

result: 
--------------------------------------------------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         20  Anjali Singh   21 Female Programming     94          97    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         26   Nandita Rao   21 Female Programming     92          96    A+
          8  Ananya Gupta   21 Female Programming     89          96     A
         14 Kavitha Rajan   21 Female Programming     86          93     A
---------------------------------------------

In [ ]:
def generate_answer(user_question, query_results_df, client, model):

    # Check if dataframe is empty
    if query_results_df is None or len(query_results_df) == 0:
        return "No results found."

    results_text = query_results_df.to_string(index=False)

    prompt = f"""
The user asked: '{user_question}'

The database returned these results:

{results_text}

Please write a clear, friendly, 2-3 sentence answer to the user's question based on these results.
Be specific. Mention actual names and numbers from the data.
Do not add information not present in the results.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()
def smart_text_to_sql_agent(user_question, conn, client, model, verbose=True):
    """
    Enhance agent that returns both a data table AND a natural language answer.
    """
    print("=" * 60)
    print(f"Question: {user_question}")
    print("=" * 60)

    schema_text = get_schema(conn)

    print("Generating SQL...")
    generated_sql = generate_sql(user_question, schema_text, client, model)
    print(f"SQL: {generated_sql}")

    result_df, error = execute_sql(generated_sql, conn)
    if error:
        print(f"SQL execution error: {error}")
        return None, None, generated_sql
    print(f"Data ({len(result_df)} rows returned):")
    print(result_df)
    print("\nGenerating natural language answer...")
    answer = generate_answer(user_question, result_df, client, model)
    print("\nAnswer:")
    print(answer)

    return result_df, answer, generated_sql

In [ ]:
result_df, nl_answer, sql_query = smart_text_to_sql_agent(
    "Who are the top 5 students in Programming?",
    conn, client, MODEL
)

Question: Who are the top 5 students in Programming?
Generating SQL...
SQL: SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
Data (5 rows returned):
           name
0  Aditya Kumar
1   Rohan Mehta
2  Anjali Singh
3   Nandita Rao
4    Arjun Nair

Generating natural language answer...

Answer:
Based on the results, the top 5 students in Programming are:

1. Aditya Kumar
2. Rohan Mehta
3. Anjali Singh
4. Nandita Rao
5. Arjun Nair

These students have secured the top spots in the programming class.
